# Relation KG Experiment (Unified)

단일 정본 관계그래프(`relation_edges_confirmed.csv`) 기준 D 실험 노트북입니다.


In [ ]:
# Optional installs
# %pip install -q openai pandas python-dotenv tqdm


In [1]:
from pathlib import Path
import os

env_path = Path.cwd() / ".env"
for line in env_path.read_text(encoding="utf-8").splitlines():
    s = line.strip()
    if s and not s.startswith("#") and "=" in s:
        k, v = s.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

print("key loaded(after):", bool(os.getenv("OPENAI_API_KEY")))

key loaded(after): True


In [3]:
import os
import subprocess
from pathlib import Path
import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

BASE = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
SCRIPTS = BASE / 'scripts'
DATA = BASE / 'data'
REL = DATA / 'relation_kg'
OUT = BASE / 'outputs'
EVAL = BASE / 'eval'

if load_dotenv is not None:
    load_dotenv(BASE / '.env', override=False)
    load_dotenv(override=False)

RUN_DATE = os.getenv('RUN_DATE', '2026-04-12')
DRY_RUN = os.getenv('DRY_RUN', 'true').lower() == 'true'

print('RUN_DATE =', RUN_DATE)
print('DRY_RUN =', DRY_RUN)
print('OPENAI_API_KEY loaded =', bool(os.getenv('OPENAI_API_KEY')))


RUN_DATE = 2026-04-12
DRY_RUN = False
OPENAI_API_KEY loaded = True


In [4]:
# 1) Optional: refresh auto candidates (not canonical)
REFRESH_AUTO = False
if REFRESH_AUTO:
    cmd = ['python3', str(SCRIPTS / 'extract_relation_candidates.py'), '--allow-cooccurrence']
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


In [5]:
# 2) Build sample relation context from canonical confirmed edges
cmd = [
    'python3', str(SCRIPTS / 'build_relation_context.py'),
    '--samples-csv', str(DATA / 'samples_tagged_v1.csv'),
    '--edges-csv', str(REL / 'relation_edges_confirmed.csv'),
    '--seed-csv', str(REL / 'character_seeds.csv'),
    '--out-csv', str(REL / 'sample_relation_context.csv')
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

ctx = pd.read_csv(REL / 'sample_relation_context.csv')
print('context rows:', len(ctx))
ctx.head(10)


Running: python3 /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/scripts/build_relation_context.py --samples-csv /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/data/samples_tagged_v1.csv --edges-csv /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/data/relation_kg/relation_edges_confirmed.csv --seed-csv /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/data/relation_kg/character_seeds.csv --out-csv /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/data/relation_kg/sample_relation_context.csv
Wrote 80 rows -> /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/data/relation_kg/sample_relation_context.csv
context rows: 80


,sample_id,sentence_type,source_text,detected_entities,relation_context
0,S001,UI,Unstoppable Force,NaN,- No explicit relation context found. Use neut...
1,S002,UI,Nemean Gauntlets,NaN,- No explicit relation context found. Use neut...
2,S003,UI,Icarus Drifter Legs,The Drifter,- Eris Morn romantic_partner_of The Drifter (c...
3,S004,UI,Light and Dark Saga Exotics,NaN,- No explicit relation context found. Use neut...
4,S005,UI,Dreaming Spectrum,NaN,- No explicit relation context found. Use neut...
5,S006,UI,Ana: Black Box,Ana Bray,- No explicit relation context found. Use neut...
6,S007,UI,Transmatter Components,NaN,- No explicit relation context found. Use neut...
7,S008,UI,The Empty Tank: Legend,NaN,- No explicit relation context found. Use neut...
8,S009,UI,Bounty Hunter: Tangled Shore,NaN,- No explicit relation context found. Use neut...
9,S010,UI,Swordmaster's Gauntlets,NaN,- No explicit relation context found. Use neut...


In [6]:
# 3) Inspect rows with explicit relation context
ctx = pd.read_csv(REL / 'sample_relation_context.csv')
has_ctx = ctx[~ctx['relation_context'].str.contains('No explicit relation context', na=False)]
print('with context:', len(has_ctx), '/', len(ctx))
has_ctx[['sample_id','sentence_type','source_text','detected_entities','relation_context']].head(20)


with context: 4 / 80


,sample_id,sentence_type,source_text,detected_entities,relation_context
2,S003,UI,Icarus Drifter Legs,The Drifter,- Eris Morn romantic_partner_of The Drifter (c...
41,S042,perk,Prosperity (Vanguard),Vanguard,- Caiatl allied_with Vanguard (confidence: hig...
53,S054,dialogue,"""Many regard the Hive as mere monsters. But th...",Eris Morn|Hive,- Eris Morn romantic_partner_of The Drifter (c...
62,S063,dialogue,"""Have you ever stared into the green soulfire ...",Eris Morn,- Eris Morn romantic_partner_of The Drifter (c...


In [7]:
# 4) Run D condition
cmd = ['python3', str(SCRIPTS / 'run_condition_d.py'), '--run-date', RUN_DATE]
if DRY_RUN:
    cmd.append('--dry-run')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

df_d = pd.read_csv(OUT / f'run_{RUN_DATE}' / 'D_outputs.csv')
print('D outputs:', len(df_d))
df_d.head(10)


Running: python3 /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/scripts/run_condition_d.py --run-date 2026-04-12
Wrote 80 rows -> /Users/yiji/Desktop/work/pseudo_lab/game_translation_exp/outputs/run_2026-04-12/D_outputs.csv
D outputs: 80


,sample_id,source_text,translation,sentence_type,relation_context
0,S001,Unstoppable Force,막을 수 없는 힘,UI,- No explicit relation context found. Use neut...
1,S002,Nemean Gauntlets,네메안 건틀릿,UI,- No explicit relation context found. Use neut...
2,S003,Icarus Drifter Legs,이카루스 드리프터 다리,UI,- Eris Morn romantic_partner_of The Drifter (c...
3,S004,Light and Dark Saga Exotics,빛과 어둠 사가 특급 장비,UI,- No explicit relation context found. Use neut...
4,S005,Dreaming Spectrum,꿈꾸는 스펙트럼,UI,- No explicit relation context found. Use neut...
5,S006,Ana: Black Box,아나: 블랙박스,UI,- No explicit relation context found. Use neut...
6,S007,Transmatter Components,트랜스매터 구성품,UI,- No explicit relation context found. Use neut...
7,S008,The Empty Tank: Legend,빈 탱크: 전설,UI,- No explicit relation context found. Use neut...
8,S009,Bounty Hunter: Tangled Shore,현상금 사냥꾼: 얽힌 해안,UI,- No explicit relation context found. Use neut...
9,S010,Swordmaster's Gauntlets,검술가의 건틀릿,UI,- No explicit relation context found. Use neut...


In [10]:
# 5) Build A/B/C/D comparison sheet
run_dir = OUT / f'run_{RUN_DATE}'
samples = pd.read_csv(DATA / 'samples.csv')

A = pd.read_csv(run_dir / 'A_outputs.csv') if (run_dir / 'A_outputs.csv').exists() else None
B = pd.read_csv(run_dir / 'B_outputs.csv') if (run_dir / 'B_outputs.csv').exists() else None
C = pd.read_csv(run_dir / 'C_outputs.csv') if (run_dir / 'C_outputs.csv').exists() else None
D = pd.read_csv(run_dir / 'D_outputs.csv') if (run_dir / 'D_outputs.csv').exists() else None

merged = samples[['sample_id','sentence_type','source_text']].copy()
if A is not None:
    merged = merged.merge(A[['sample_id','translation']].rename(columns={'translation':'condition_A'}), on='sample_id', how='left')
if B is not None:
    merged = merged.merge(B[['sample_id','translation']].rename(columns={'translation':'condition_B'}), on='sample_id', how='left')
if C is not None:
    merged = merged.merge(C[['sample_id','translation']].rename(columns={'translation':'condition_C'}), on='sample_id', how='left')
if D is not None:
    merged = merged.merge(D[['sample_id','translation']].rename(columns={'translation':'condition_D'}), on='sample_id', how='left')

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'D_meaning','D_term','D_consistency','D_naturalness','D_ui_fit',
    'best','error_type','notes'
]:
    if col not in merged.columns:
        merged[col] = ''

out_eval = EVAL / f'eval_sheet_prefilled_ABCD_{RUN_DATE}.csv'
merged.to_csv(out_eval, index=False, encoding='utf-8')
print('saved:', out_eval)
print('rows:', len(merged))


ValueError: You are trying to merge on str and int64 columns for key 'sample_id'. If you wish to proceed you should use pd.concat

## Notes
- Canonical source: `data/relation_kg/relation_edges_confirmed.csv`
- `.env`는 `game_translation_exp/.env`와 현재 작업 디렉터리에서 로딩합니다.
